# Model C — TTA embeddings + CE

Overlapping-window embeddings at inference time. Kaggle public LB: 0.798.


In [1]:
from google.colab import drive
drive.mount("/content/drive")

import importlib.util
import os
import sys
from pathlib import Path

def _load_project():
    my_drive = Path("/content/drive/MyDrive")
    init_candidates = [
        p for p in my_drive.rglob("colab_init.py")
        if (p.parent / "birdclef").is_dir()
    ]
    if init_candidates:
        init_path = min(init_candidates, key=lambda p: len(p.parts))
        spec = importlib.util.spec_from_file_location("colab_init", init_path)
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        return
    roots = [
        nb.parent.resolve()
        for nb in my_drive.rglob("02_download_and_extract_embeddings.ipynb")
        if (nb.parent / "birdclef").is_dir()
    ]
    if not roots:
        raise FileNotFoundError(
            "Unzip the full repository on Google Drive, then open a notebook from that folder."
        )
    root = min(roots, key=lambda p: len(p.parts))
    sys.path.insert(0, str(root))
    os.chdir(root)

_load_project()

!pip install -q onnxscript onnxruntime-gpu soundfile tqdm scikit-learn matplotlib seaborn kaggle

from birdclef.colab import colab_prepare_training

colab_prepare_training(stage_tta=True)


Mounted at /content/drive
Working directory: /content/drive/MyDrive/BirdCLEF_Project/repro
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.0/277.0 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 11.3 MB/s eta 0:00:00
Copied train.csv to /content/train.csv
Copied sample_submission.csv to /content/sample_submission.csv
Unzipped baseline embeddings to /content/embeddings_v2
Unzipped TTA embeddings to /content/embeddings_v2_TTA
Project root: /content/drive/MyDrive/BirdCLEF_Project/repro


In [2]:
import os
import json
import warnings
import logging
from contextlib import redirect_stdout, redirect_stderr

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader

from birdclef import (
    BirdDataset,
    BirdMoE,
    FocalLoss,
    competition_macro_roc_auc,
    plot_and_save_learning_curves,
    prepare_baseline_data,
    prepare_tta_data,
    seed_everything,
)
import birdclef.paths as paths

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")


Training on device: cuda


In [3]:
df_TTA, NUM_CLASSES, _ = prepare_tta_data()
model = BirdMoE(input_dim=1536, num_classes=NUM_CLASSES).to(device)


In [4]:
SAVE_DIR = paths.MODELS_DIR / "tta_ce_10ep"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
EPOCHS = 10
tta_fold_scores = []

for TRAIN_FOLD in range(5):
    best_auc = 0.0
    train_df = df_TTA[df_TTA["fold"] != TRAIN_FOLD].reset_index(drop=True)
    valid_df = df_TTA[df_TTA["fold"] == TRAIN_FOLD].reset_index(drop=True)
    train_ds = BirdDataset(train_df, is_tta=True)
    valid_ds = BirdDataset(valid_df, is_tta=True)
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0)
    valid_loader = DataLoader(valid_ds, batch_size=64, shuffle=False, num_workers=0)
    fold_train_losses = []
    fold_val_losses = []
    fold_val_aucs = []

    print(f"TRAINING FOLD {TRAIN_FOLD}")
    print(f"Training on {len(train_df)} samples, validating on {len(valid_df)} samples")

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        avg_train_loss = train_loss / len(train_loader)

        model.eval()
        val_loss = 0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for inputs, labels in valid_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                all_preds.append(F.softmax(outputs, dim=1).cpu().numpy())
                all_labels.append(labels.cpu().numpy())
        avg_val_loss = val_loss / len(valid_loader)
        val_preds = np.concatenate(all_preds)
        val_labels = np.concatenate(all_labels)
        val_labels_onehot = np.eye(NUM_CLASSES)[val_labels]
        current_auc = competition_macro_roc_auc(val_labels_onehot, val_preds)
        print(
            f"Epoch {epoch + 1:02d}/{EPOCHS} | "
            f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val AUC: {current_auc:.4f}"
        )
        fold_train_losses.append(avg_train_loss)
        fold_val_losses.append(avg_val_loss)
        fold_val_aucs.append(current_auc)
        if current_auc > best_auc:
            best_auc = current_auc
            save_path = SAVE_DIR / f"best_moe_fold{TRAIN_FOLD}.pth"
            torch.save(model.state_dict(), save_path)
            print(f"Model saved to {save_path}")

    onnx_save_path = os.path.join(str(SAVE_DIR), f"bird_moe_fold{TRAIN_FOLD}.onnx")
    plot_and_save_learning_curves(fold_train_losses, fold_val_losses, fold_val_aucs, TRAIN_FOLD, str(SAVE_DIR))
    model.load_state_dict(torch.load(save_path, map_location=device))
    model.eval()
    dummy_input = torch.randn(1, 1536).to(device)
    with open(os.devnull, "w") as f, redirect_stdout(f), redirect_stderr(f), warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for name in ("onnx", "onnxruntime", "torch.onnx", "onnxscript"):
            logging.getLogger(name).setLevel(logging.CRITICAL)
        torch.onnx.export(
            model,
            dummy_input,
            onnx_save_path,
            export_params=True,
            opset_version=15,
            do_constant_folding=True,
            input_names=["embedding"],
            output_names=["logits"],
            dynamic_axes={"embedding": {0: "batch_size"}, "logits": {0: "batch_size"}},
        )
    print(f"Exported ONNX model to {onnx_save_path}")
    tta_fold_scores.append(best_auc)

print(f"CV score: {np.mean(tta_fold_scores):.4f} (+/- {np.std(tta_fold_scores):.4f})")


100%|██████████| 7110/7110 [00:09<00:00, 732.84it/s]


TRAINING FOLD 0
Training on 28439 samples, validating on 7110 samples
Epoch 01/10 | Train Loss: 0.9640 | Val Loss: 1.4545 | Val AUC: 0.9765
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/tta_ce_10ep/best_moe_fold0.pth
Epoch 02/10 | Train Loss: 0.7349 | Val Loss: 1.4850 | Val AUC: 0.9747
Epoch 03/10 | Train Loss: 0.6823 | Val Loss: 1.5078 | Val AUC: 0.9757
Epoch 04/10 | Train Loss: 0.6468 | Val Loss: 1.5259 | Val AUC: 0.9762
Epoch 05/10 | Train Loss: 0.6242 | Val Loss: 1.4899 | Val AUC: 0.9769
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/tta_ce_10ep/best_moe_fold0.pth
Epoch 06/10 | Train Loss: 0.6089 | Val Loss: 1.5384 | Val AUC: 0.9750
Epoch 07/10 | Train Loss: 0.5956 | Val Loss: 1.5459 | Val AUC: 0.9760
Epoch 08/10 | Train Loss: 0.5881 | Val Loss: 1.5532 | Val AUC: 0.9758
Epoch 09/10 | Train Loss: 0.5801 | Val Loss: 1.5381 | Val AUC: 0.9757
Epoch 10/10 | Train Loss: 0.5736 | Val Loss: 1.5167 | Val AUC: 0.9764
Learning curves 

100%|██████████| 7110/7110 [00:09<00:00, 727.60it/s]


TRAINING FOLD 1
Training on 28439 samples, validating on 7110 samples
Epoch 01/10 | Train Loss: 0.6915 | Val Loss: 1.1085 | Val AUC: 0.9881
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/tta_ce_10ep/best_moe_fold1.pth
Epoch 02/10 | Train Loss: 0.6277 | Val Loss: 1.2321 | Val AUC: 0.9843
Epoch 03/10 | Train Loss: 0.6046 | Val Loss: 1.3045 | Val AUC: 0.9821
Epoch 04/10 | Train Loss: 0.5885 | Val Loss: 1.3457 | Val AUC: 0.9812
Epoch 05/10 | Train Loss: 0.5821 | Val Loss: 1.3690 | Val AUC: 0.9798
Epoch 06/10 | Train Loss: 0.5736 | Val Loss: 1.4011 | Val AUC: 0.9792
Epoch 07/10 | Train Loss: 0.5678 | Val Loss: 1.3915 | Val AUC: 0.9789
Epoch 08/10 | Train Loss: 0.5623 | Val Loss: 1.4178 | Val AUC: 0.9793
Epoch 09/10 | Train Loss: 0.5575 | Val Loss: 1.4266 | Val AUC: 0.9772
Epoch 10/10 | Train Loss: 0.5545 | Val Loss: 1.4319 | Val AUC: 0.9776
Learning curves saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/tta_ce_10ep/learning_curves_fold1.png

100%|██████████| 7110/7110 [00:09<00:00, 722.54it/s]


TRAINING FOLD 2
Training on 28439 samples, validating on 7110 samples
Epoch 01/10 | Train Loss: 0.6790 | Val Loss: 1.0491 | Val AUC: 0.9883
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/tta_ce_10ep/best_moe_fold2.pth
Epoch 02/10 | Train Loss: 0.6222 | Val Loss: 1.1985 | Val AUC: 0.9837
Epoch 03/10 | Train Loss: 0.6011 | Val Loss: 1.2956 | Val AUC: 0.9819
Epoch 04/10 | Train Loss: 0.5866 | Val Loss: 1.3378 | Val AUC: 0.9802
Epoch 05/10 | Train Loss: 0.5796 | Val Loss: 1.3516 | Val AUC: 0.9796
Epoch 06/10 | Train Loss: 0.5719 | Val Loss: 1.3656 | Val AUC: 0.9798
Epoch 07/10 | Train Loss: 0.5678 | Val Loss: 1.3655 | Val AUC: 0.9802
Epoch 08/10 | Train Loss: 0.5632 | Val Loss: 1.3935 | Val AUC: 0.9772
Epoch 09/10 | Train Loss: 0.5607 | Val Loss: 1.4092 | Val AUC: 0.9790
Epoch 10/10 | Train Loss: 0.5563 | Val Loss: 1.4107 | Val AUC: 0.9774
Learning curves saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/tta_ce_10ep/learning_curves_fold2.png

100%|██████████| 7110/7110 [00:10<00:00, 673.18it/s]


TRAINING FOLD 3
Training on 28439 samples, validating on 7110 samples
Epoch 01/10 | Train Loss: 0.6581 | Val Loss: 1.0808 | Val AUC: 0.9904
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/tta_ce_10ep/best_moe_fold3.pth
Epoch 02/10 | Train Loss: 0.6069 | Val Loss: 1.2147 | Val AUC: 0.9857
Epoch 03/10 | Train Loss: 0.5889 | Val Loss: 1.2867 | Val AUC: 0.9824
Epoch 04/10 | Train Loss: 0.5777 | Val Loss: 1.3223 | Val AUC: 0.9827
Epoch 05/10 | Train Loss: 0.5697 | Val Loss: 1.3630 | Val AUC: 0.9826
Epoch 06/10 | Train Loss: 0.5648 | Val Loss: 1.3778 | Val AUC: 0.9811
Epoch 07/10 | Train Loss: 0.5590 | Val Loss: 1.4098 | Val AUC: 0.9781
Epoch 08/10 | Train Loss: 0.5552 | Val Loss: 1.4137 | Val AUC: 0.9781
Epoch 09/10 | Train Loss: 0.5527 | Val Loss: 1.4083 | Val AUC: 0.9780
Epoch 10/10 | Train Loss: 0.5494 | Val Loss: 1.4823 | Val AUC: 0.9768
Learning curves saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/tta_ce_10ep/learning_curves_fold3.png

100%|██████████| 7109/7109 [00:10<00:00, 680.92it/s]


TRAINING FOLD 4
Training on 28440 samples, validating on 7109 samples
Epoch 01/10 | Train Loss: 0.6530 | Val Loss: 0.9882 | Val AUC: 0.9921
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/tta_ce_10ep/best_moe_fold4.pth
Epoch 02/10 | Train Loss: 0.6061 | Val Loss: 1.1258 | Val AUC: 0.9881
Epoch 03/10 | Train Loss: 0.5897 | Val Loss: 1.2158 | Val AUC: 0.9855
Epoch 04/10 | Train Loss: 0.5784 | Val Loss: 1.2552 | Val AUC: 0.9850
Epoch 05/10 | Train Loss: 0.5705 | Val Loss: 1.3030 | Val AUC: 0.9825
Epoch 06/10 | Train Loss: 0.5666 | Val Loss: 1.3036 | Val AUC: 0.9824
Epoch 07/10 | Train Loss: 0.5615 | Val Loss: 1.3229 | Val AUC: 0.9811
Epoch 08/10 | Train Loss: 0.5582 | Val Loss: 1.3388 | Val AUC: 0.9811
Epoch 09/10 | Train Loss: 0.5552 | Val Loss: 1.3681 | Val AUC: 0.9807
Epoch 10/10 | Train Loss: 0.5535 | Val Loss: 1.3754 | Val AUC: 0.9806
Learning curves saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/tta_ce_10ep/learning_curves_fold4.png